In [ ]:
#####################################################
#
# APLICAR Regresión lineal a datos NUEVOS preprocesados con PCA
#
#####################################################
# Deben cargarse los archivos
# - T_new_final.csv (son los datos nuevos ya preprocesados con PCA)
# - expected_columns.json
# - modelo_reg_lineal.pkl
#
# Devolverá
# Regresion_lineal_nuevos_predicciones.csv: csv de T_new_final con las predicciones de la regresión lineal
#####################################################

# ===== Carga del modelo y predicción en datos nuevos =====
import pandas as pd
import joblib, json

# 1) Cargar artefactos
modelo = joblib.load("modelo_reg_lineal.pkl")
with open("expected_columns.json", "r", encoding="utf-8") as f:
    expected_cols = json.load(f)["columns"]

# 2) Preparar X con el mismo esquema que en entrenamiento
def preparar_X_nuevo(df_nuevo: pd.DataFrame) -> pd.DataFrame:
    # si el CSV trae la última columna como 'charges' u objetivo, se ignora
    posibles_obj = {"charges", "target", "y"}
    cols_a_quitar = [c for c in df_nuevo.columns if c in posibles_obj]
    df_nuevo = df_nuevo.drop(columns=cols_a_quitar, errors="ignore")

    # agregar columnas faltantes como 0 y ordenar como en train
    X = df_nuevo.reindex(columns=expected_cols, fill_value=0)

    # descartar columnas extra no vistas en entrenamiento (por si acaso)
    X = X.loc[:, expected_cols]
    return X

# 3) Usar en un lote nuevo
df_nuevo = pd.read_csv("T_new_final.csv")
X_nuevo = preparar_X_nuevo(df_nuevo)
y_pred  = modelo.predict(X_nuevo)
print(y_pred[:10])


In [ ]:
reg_new_final = pd.concat([df_nuevo, pd.DataFrame(y_pred, columns=["yhat"])], axis=1)
reg_new_final.to_csv("Regresion_lineal_nuevos_predicciones.csv",index=False)

In [ ]:
reg_new_final